# PhoBERT V5 - Sieu cap cam xuc tieng Viet (Toi uu GPU T4) (Dùng Colab mail B để finetune và lưu kết quả vào Drive của mail A)

Muc tieu cua V5:
- Tang do chinh xac va F1 macro
- Tang do tu tin (co hieu chinh calibration)
- Xu ly tot cau cam xuc manh, mia mai, cham biem
- Fine-tune nhanh tren Google Colab T4

Ky thuat chinh:
- Hardcase + sarcasm augmentation
- Focal Loss + Label Smoothing + Class Weights
- Dynamic padding + fp16 + group_by_length
- Temperature scaling cho do tu tin
- Truc quan hoa day du bang bieu do va ma tran nham lan (tieng Viet)


*LƯU Ý: Phiên bản v5 này là bản nâng cấp trực tiếp từ phiên bản "phobert-v3". Phiên bản v3 là kết quả của việc chạy 2 notebook phobert-v3-1st.ipynb và phobert-v3-2nd.ipynb.

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn matplotlib seaborn

In [ ]:
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LABEL_MAP = {0: 'Tiêu cực', 1: 'Tích cực', 2: 'Trung tính'}
ID2LABEL = {k: v for k, v in LABEL_MAP.items()}
LABEL2ID = {v: k for k, v in LABEL_MAP.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Chay bang mail B, nhung doc/ghi trong thu muc share tu mail A
ROOT_CANDIDATES = [
    '/content/drive/MyDrive/DO_AN_1_main',
]

ROOT = None
for p in ROOT_CANDIDATES:
    if os.path.exists(p):
        ROOT = p
        break

if ROOT is None:
    shortcut_root = '/content/drive/MyDrive/.shortcut-targets-by-id'
    if os.path.exists(shortcut_root):
        for base, dirs, files in os.walk(shortcut_root):
            if base.endswith('/DO_AN_1_main'):
                ROOT = base
                break

if ROOT is None:
    raise FileNotFoundError(
        'Khong tim thay thu muc DO_AN_1_main da duoc share. '
        'Hay Add shortcut thu muc share vao MyDrive truoc.'
    )

DATA_PATH = f'{ROOT}/data/data_v4_tu_crawl/data_cleaned_finetune.csv'
BASE_MODEL = f'{ROOT}/phobert-v3'
OUT_DIR = f'{ROOT}/phobert-v5'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(BASE_MODEL):
    raise FileNotFoundError(f'Khong tim thay model V3 tai: {BASE_MODEL}')

print('ROOT dang dung:', ROOT)
print('Data path ton tai:', os.path.exists(DATA_PATH))
print('Model khoi tao (co dinh):', BASE_MODEL)
print('Thu muc luu:', OUT_DIR)

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={'Review': 'text', 'Label_number': 'label', 'Label_text': 'label_text'})
df = df[['text', 'label', 'label_text']].copy()
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

def clean_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

df['text'] = df['text'].map(clean_text)
df = df[df['text'].str.len() > 2].reset_index(drop=True)

print(f'Tong so mau goc: {len(df):,}')
print(df['label'].value_counts().sort_index())
display(df.head(5))

In [ ]:
SARC_NEG = [
    'tuyet voi qua, lan sau khong bao gio mua nua',
    '10 diem cho su that vong',
    'cam on shop da cho trai nghiem te nhat',
    'hang ngon lam, ngon den muc hong ngay',
    '5 sao vi khong co 0 sao',
]
STRONG_NEG = [
    'lua dao', 'do rac', 'phi tien', 'khong dung duoc', 'qua te',
    'hang hong ngay', 'khong bao gio mua lai', 'mat tien oan',
]
STRONG_POS = [
    'qua xuat sac', 'rat dang tien', 'chat luong vuot mong doi',
    'se mua lai nhieu lan', 'rat hai long', 'tot ngoai mong doi',
]
NEU_HARD = [
    'da nhan hang, chua dung nen chua danh gia duoc',
    'giao hang dung hen, dong goi on',
    'tam duoc, can them thoi gian su dung',
    'binh thuong, khong tot cung khong xau',
]

AUG_REPEAT = 10
aug_rows = []
for _ in range(AUG_REPEAT):
    aug_rows.extend([{'text': t, 'label': 0, 'label_text': 'neg'} for t in SARC_NEG + STRONG_NEG])
    aug_rows.extend([{'text': t, 'label': 1, 'label_text': 'pos'} for t in STRONG_POS])
    aug_rows.extend([{'text': t, 'label': 2, 'label_text': 'neu'} for t in NEU_HARD])

df_aug = pd.DataFrame(aug_rows)
df_all = pd.concat([df, df_aug], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f'Mau bo sung: {len(df_aug):,}')
print(f'Tong sau bo sung: {len(df_all):,}')
print(df_all['label'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 5))
counts = df_all['label'].value_counts().sort_index()
labels_vi = [LABEL_MAP[i] for i in counts.index]
sns.barplot(x=labels_vi, y=counts.values, palette='viridis')
plt.title('Phân bố số lượng mẫu theo nhãn (V5)', fontsize=13, fontweight='bold')
plt.xlabel('Nhóm cảm xúc')
plt.ylabel('Số lượng mẫu')
for i, v in enumerate(counts.values):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    df_all, test_size=0.2, random_state=SEED, stratify=df_all['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label']
)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('Class weight:', {LABEL_MAP[i]: float(w) for i, w in enumerate(class_weights)})

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
MAX_LEN = 128

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize_batch, batched=True, num_proc=2)
val_ds = val_ds.map(tokenize_batch, batched=True, num_proc=2)
test_ds = test_ds.map(tokenize_batch, batched=True, num_proc=2)

cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

In [ ]:
class FocalLabelSmoothingLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            reduction='none',
            label_smoothing=self.label_smoothing,
        )
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class V5Trainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, label_smoothing=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = FocalLabelSmoothingLoss(
            class_weights=class_weights,
            gamma=gamma,
            label_smoothing=label_smoothing,
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels') if 'labels' in inputs else inputs.pop('label')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

In [ ]:
import inspect

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

# warmup_ratio da bi canh bao deprecated, doi sang warmup_steps de on dinh ve sau
steps_per_epoch = max(1, len(train_ds) // (16 * 2))
total_steps = steps_per_epoch * 5
warmup_steps = max(1, int(0.08 * total_steps))

ta_kwargs = dict(
    output_dir='./results_v5',
    num_train_epochs=5,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=80,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

# Tuong thich nhieu phien ban TrainingArguments
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'group_by_length' in ta_sig:
    ta_kwargs['group_by_length'] = True
else:
    print("Canh bao: transformers hien tai khong ho tro 'group_by_length', bo qua de tranh loi.")

train_args = TrainingArguments(**ta_kwargs)

# Tuong thich nhieu phien ban Trainer (tokenizer/processing_class)
trainer_kwargs = dict(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    gamma=2.2,
    label_smoothing=0.07,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_sig = inspect.signature(Trainer.__init__).parameters
if 'tokenizer' in trainer_sig:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_sig:
    trainer_kwargs['processing_class'] = tokenizer
else:
    print("Canh bao: Trainer hien tai khong nhan tokenizer/processing_class, tiep tuc khong truyen tham so nay.")

trainer = V5Trainer(**trainer_kwargs)

print('San sang huan luyen V5 tren T4.')

In [ ]:
train_result = trainer.train()
print('Thoi gian train (giay):', round(train_result.metrics.get('train_runtime', 0), 2))
print('Mau/giay:', round(train_result.metrics.get('train_samples_per_second', 0), 2))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history = history.fillna(method='ffill')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'loss' in history:
    axes[0].plot(history.index, history['loss'], label='Train loss', color='#1f77b4')
if 'eval_loss' in history:
    axes[0].plot(history.index, history['eval_loss'], label='Eval loss', color='#ff7f0e')
axes[0].set_title('Đường cong mất mát', fontweight='bold')
axes[0].set_xlabel('Bước log')
axes[0].set_ylabel('Giá trị loss')
axes[0].legend()

if 'eval_accuracy' in history:
    axes[1].plot(history.index, history['eval_accuracy'], label='Độ chính xác', color='#2ca02c')
if 'eval_f1' in history:
    axes[1].plot(history.index, history['eval_f1'], label='F1 macro', color='#d62728')
axes[1].set_title('Tiến trình chất lượng đánh giá', fontweight='bold')
axes[1].set_xlabel('Bước log')
axes[1].set_ylabel('Điểm số')
axes[1].legend()

plt.suptitle('Thống kê huấn luyện PhoBERT V5', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pred_raw = trainer.predict(test_ds)
logits = pred_raw.predictions
y_true = pred_raw.label_ids
y_pred = np.argmax(logits, axis=1)

acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
print(f'Độ chính xác test: {acc*100:.2f}%')
print(f'Precision macro: {p*100:.2f}%')
print(f'Recall macro: {r*100:.2f}%')
print(f'F1 macro: {f1*100:.2f}%')

print('\nBáo cáo phân loại chi tiết:')
print(classification_report(y_true, y_pred, target_names=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]], digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]],
    yticklabels=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]],
)
plt.title('Ma trận nhầm lẫn - PhoBERT V5', fontsize=13, fontweight='bold')
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thực tế')
plt.tight_layout()
plt.show()

In [ ]:
class ModelWithTemperature(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, **inputs):
        logits = self.model(**inputs).logits
        return logits / self.temperature.clamp(min=1e-3)

def tune_temperature(trainer_obj, valid_ds, max_iter=50):
    wrapped = ModelWithTemperature(trainer_obj.model).to(device)
    wrapped.eval()

    val_pred = trainer_obj.predict(valid_ds)
    logits_val = torch.tensor(val_pred.predictions, dtype=torch.float32, device=device)
    labels_val = torch.tensor(val_pred.label_ids, dtype=torch.long, device=device)

    optimizer = torch.optim.LBFGS([wrapped.temperature], lr=0.05, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits_val / wrapped.temperature.clamp(min=1e-3), labels_val)
        loss.backward()
        return loss

    optimizer.step(closure)
    return wrapped.temperature.detach().item()

best_temp = tune_temperature(trainer, val_ds)
print(f'Nhiet do hieu chinh toi uu: {best_temp:.4f}')

In [ ]:
def predict_v5(text, temperature=1.0):
    model.eval()
    encoded = tokenizer(text.lower().strip(), return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = model(**encoded).logits / max(temperature, 1e-3)
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))
    confidence = float(np.max(probs))
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return pred_id, confidence, entropy, probs

test_sentences = [
    'shop qua tuyet voi, giao sai mau nhung van bat minh cho 5 sao',
    'hang dep xuat sac, dung mo ta, rat dang tien',
    'tam duoc, moi nhan nen chua danh gia sau',
    'khong te lam, nhung cung khong co gi dac biet',
    'doi 3 ngay la hong, dung la qua chat luong roi do',
    'cam on shop, trai nghiem nay qua te va minh se khong quay lai',
]

for s in test_sentences:
    pid, conf, ent, probs = predict_v5(s, temperature=best_temp)
    print('-' * 90)
    print('Cau:', s)
    print('Du doan:', LABEL_MAP[pid], f'| Do tu tin: {conf*100:.2f}% | Entropy: {ent:.4f}')
    print('Xac suat [Tieu cuc, Tich cuc, Trung tinh]:', np.round(probs, 4))

In [ ]:
v3_f1 = 0.0  # Điền F1 V3
v4_f1 = 0.0  # Điền F1 V4
v5_f1 = float(f1)

v3_acc = 0.0  # Điền Acc V3
v4_acc = 0.0  # Điền Acc V4
v5_acc = float(acc)

df_cmp = pd.DataFrame({
    'Phiên bản': ['V3', 'V4', 'V5'],
    'F1_macro': [v3_f1, v4_f1, v5_f1],
    'Độ_chính_xác': [v3_acc, v4_acc, v5_acc],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=df_cmp, x='Phiên bản', y='F1_macro', ax=axes[0], palette='Set2')
axes[0].set_title('So sánh F1 macro giữa V3/V4/V5', fontweight='bold')
axes[0].set_xlabel('Phiên bản')
axes[0].set_ylabel('Điểm F1 macro')

sns.barplot(data=df_cmp, x='Phiên bản', y='Độ_chính_xác', ax=axes[1], palette='Set3')
axes[1].set_title('So sánh độ chính xác giữa V3/V4/V5', fontweight='bold')
axes[1].set_xlabel('Phiên bản')
axes[1].set_ylabel('Độ chính xác')

for ax in axes:
    for pz in ax.patches:
        h = pz.get_height()
        ax.annotate(f'{h:.3f}', (pz.get_x() + pz.get_width() / 2, h), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

metrics = {
    'version': 'V5',
    'base_model': BASE_MODEL,
    'test_accuracy': float(acc),
    'test_precision_macro': float(p),
    'test_recall_macro': float(r),
    'test_f1_macro': float(f1),
    'temperature': float(best_temp),
    'max_len': MAX_LEN,
    'epochs': int(train_args.num_train_epochs),
    'learning_rate': float(train_args.learning_rate),
    'batch_train': int(train_args.per_device_train_batch_size),
    'batch_eval': int(train_args.per_device_eval_batch_size),
}

with open(os.path.join(OUT_DIR, 'metrics_v5.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Da luu model/tokenizer/metrics vao:', OUT_DIR)
print(json.dumps(metrics, ensure_ascii=False, indent=2))